# Zero-Shot Prompting Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/zero-shot-prompting.ipynb)

## Overview

This tutorial provides a comprehensive introduction to zero-shot prompting, a powerful technique in prompt engineering that allows language models to perform tasks without specific examples or prior training. We'll explore how to design effective zero-shot prompts and implement strategies using HuggingFace Transformers with local open-source models running in Google Colab.

## Motivation

Zero-shot prompting is crucial in modern AI applications as it enables language models to generalize to new tasks without the need for task-specific training data or fine-tuning. This capability significantly enhances the flexibility and applicability of AI systems, allowing them to adapt to a wide range of scenarios and user needs with minimal setup.

## Key Components

1. **Understanding Zero-Shot Learning**: An introduction to the concept and its importance in AI.
2. **Prompt Design Principles**: Techniques for crafting effective zero-shot prompts.
3. **Task Framing**: Methods to frame various tasks for zero-shot performance.
4. **HuggingFace Transformers Integration**: Using open-source models locally for zero-shot tasks.
5. **Chat Template Format**: Leveraging system/user message separation for structured zero-shot prompting.

## Method Details

The tutorial will cover several methods for implementing zero-shot prompting:

1. **Direct Task Specification**: Crafting prompts that clearly define the task without examples.
2. **Role-Based Prompting**: Assigning specific roles to the AI to guide its responses.
3. **Format Specification**: Providing output format guidelines in the prompt.
4. **Multi-step Reasoning**: Breaking down complex tasks into simpler zero-shot steps.
5. **Comparative Analysis**: Evaluating different zero-shot prompt structures for the same task.

Throughout the tutorial, we'll use Python code with HuggingFace Transformers to demonstrate these techniques practically. We use the chat template format where the **system message** defines the task and the **user message** provides the input. The `add_generation_prompt=True` parameter ensures the model knows it should generate a response.

## Conclusion

By the end of this tutorial, learners will have gained:

1. A solid understanding of zero-shot prompting and its applications.
2. Practical skills in designing effective zero-shot prompts for various tasks.
3. Experience in implementing zero-shot techniques using HuggingFace Transformers with local models.
4. Insights into the strengths and limitations of zero-shot approaches.
5. A foundation for further exploration and innovation in prompt engineering.

This knowledge will empower learners to leverage AI models more effectively across a wide range of applications, enhancing their ability to solve novel problems and create more flexible AI systems.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Overview**
- Understand **Motivation**
- Understand **Key Components**
- Understand **Method Details**
- Understand **Conclusion**


## Setup

Let's start by installing the necessary libraries and loading an open-source model. We use 4-bit quantization to fit the model in Google Colab's GPU memory, and cache the model weights on Google Drive to avoid re-downloading.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Why Zero-Shot Works: Emergent Task Understanding

During pretraining on trillions of tokens drawn from books, websites, code, and academic papers, large language models implicitly learn the **structure** of thousands of tasks. The model never receives explicit "this is a classification task" labels — instead, it absorbs patterns from contexts like:

- *"The review is positive."* following a movie review
- *"The sentiment of the text is: negative"* in annotated datasets scraped from the web
- *"Translate the following English text to French: ..."* in parallel corpora

When you write a zero-shot prompt such as **"Classify this as positive or negative: ..."**, you are crafting a token sequence that the model has seen variants of thousands of times during pretraining. The phrase "Classify ... as positive or negative" doesn't make the model *understand* classification — it **shifts the probability distribution** over next tokens toward classification-like outputs (e.g., "Positive", "Negative") because those tokens statistically followed similar prefixes in the training data.

### Key Insight

Zero-shot works because **task descriptions activate learned representations**. The instruction tokens serve as a *soft retrieval key* into the model's compressed knowledge of task formats. The more closely your prompt resembles patterns the model saw during pretraining, the better zero-shot performance you'll get.

In [ ]:
import torch

def show_top_tokens(text, top_k=15):
    """Show the model's top-k predicted next tokens for a given text."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f"{'Rank':<6}{'Token':<25}{'Probability':>12}")
    print("-" * 45)
    for i, (p, idx) in enumerate(zip(top_probs, top_indices)):
        token = tokenizer.decode([idx.item()])
        bar = "\u2588" * int(p.item() * 40)
        print(f"{i+1:<6}{token!r:<25}{p.item():>12.4f}  {bar}")
    print()

# --- WITH task framing ---
# The instruction "Classify ... as positive or negative" activates
# classification-related representations in the model.
framed_messages = [
    {"role": "user", "content": (
        "Classify the following text as positive or negative.\n\n"
        "Text: I love this movie!\n\nSentiment:"
    )}
]
framed_text = tokenizer.apply_chat_template(
    framed_messages, tokenize=False,
    add_generation_prompt=True, enable_thinking=False
)

print("=" * 55)
print("WITH task framing (classification instruction)")
print("=" * 55)
show_top_tokens(framed_text)

# --- WITHOUT task framing ---
# The same text without classification instruction produces
# generic conversational continuations instead.
unframed_messages = [
    {"role": "user", "content": "I love this movie!"}
]
unframed_text = tokenizer.apply_chat_template(
    unframed_messages, tokenize=False,
    add_generation_prompt=True, enable_thinking=False
)

print("=" * 55)
print("WITHOUT task framing (no classification instruction)")
print("=" * 55)
show_top_tokens(unframed_text)

### How Attention Mediates This Effect

The transformer's self-attention mechanism is the engine behind this behavior. When the model processes a prompt like *"Classify the following as positive or negative: 'I love this'"*, here is what happens at the output position (where the model predicts the next token):

1. **Attention to instruction tokens**: The output position attends heavily to the tokens "Classify", "positive", "negative" — these tokens define *what kind of output is expected*.
2. **Attention to content tokens**: Simultaneously, it attends to "I", "love", "this" — these tokens provide *what to classify*.
3. **Combined signal**: The attention-weighted combination of instruction context and content context produces a hidden state that maps to classification labels in the vocabulary space.

Without the instruction tokens, step (1) is absent — the model has no signal telling it to classify, so the output position attends only to the content and produces a generic continuation (e.g., "I love this movie too!" or "That's great!").

> **This is why prompt wording matters so much in zero-shot**: different instruction phrasings activate different attention patterns, leading to different output distributions — even for the same underlying task.

## Inspecting the Model's Uncertainty

Understanding *how confident* the model is on a zero-shot task is just as important as seeing the output. We can peek inside the model by examining the **probability distribution** over candidate output tokens before generation. High probability on a single label means the model is confident; a flat distribution means it's uncertain.

In [ ]:
def inspect_confidence(text, labels):
    """Show the model's confidence over candidate classification labels."""
    task_prompt = (
        "Classify this text as positive or negative. "
        "Reply with a single word.\n\n"
        "Text: {text}\n\nSentiment:"
    )
    messages = [{"role": "user", "content": task_prompt.format(text=text)}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)

    print(f"Text: {text!r}")
    label_total = 0.0
    for label in labels:
        token_id = tokenizer.encode(label, add_special_tokens=False)[0]
        p = probs[token_id].item()
        label_total += p
        bar = "\u2588" * int(p * 50)
        print(f"  P('{label}')  {p:>8.4f}  {bar}")
    print(f"  Combined label mass: {label_total:.4f}\n")

labels = ["Positive", "Negative", "positive", "negative", "Neutral", "neutral"]

print("=== EASY CASES (high confidence expected) ===\n")
inspect_confidence("I absolutely love this product! Best purchase ever!", labels)
inspect_confidence("Terrible experience. Complete waste of money.", labels)

print("=== AMBIGUOUS CASES (lower confidence expected) ===\n")
inspect_confidence("The product arrived on time.", labels)
inspect_confidence("It was okay I guess, not great not terrible.", labels)

In [ ]:
def compare_logits(texts, labels_pair=("Positive", "Negative")):
    """Compare raw logit values for two candidate labels across texts."""
    task_prompt = (
        "Classify this text as positive or negative. "
        "Reply with a single word.\n\n"
        "Text: {text}\n\nSentiment:"
    )
    pos_label, neg_label = labels_pair
    pos_id = tokenizer.encode(pos_label, add_special_tokens=False)[0]
    neg_id = tokenizer.encode(neg_label, add_special_tokens=False)[0]

    print(f"{'Text (truncated)':<45} {'Logit(+)':>9} {'Logit(-)':>9}"
          f" {'P(+)':>7} {'P(-)':>7} {'Gap':>7} {'Pred':>8}")
    print("-" * 100)

    for text in texts:
        messages = [{"role": "user", "content": task_prompt.format(text=text)}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)

        pl = logits[pos_id].item()
        nl = logits[neg_id].item()
        pp = probs[pos_id].item()
        np_ = probs[neg_id].item()
        gap = abs(pl - nl)
        pred = pos_label if pl > nl else neg_label

        short = (text[:42] + "...") if len(text) > 42 else text
        print(f"{short:<45} {pl:>9.3f} {nl:>9.3f}"
              f" {pp:>7.4f} {np_:>7.4f} {gap:>7.2f} {pred:>8}")

test_texts = [
    "This is the best day of my life!",
    "I absolutely hate this product.",
    "The weather is nice today.",
    "It was okay, nothing special.",
    "The food was decent but the service was slow.",
    "Outstanding performance, truly remarkable!",
    "I regret buying this, total disappointment.",
    "It works as expected, no complaints.",
]

compare_logits(test_texts)
print()
print("Observe: easy cases have large logit gaps (high confidence),")
print("while ambiguous cases have small gaps (low confidence).")

### What Uncertainty Tells Us

The results above reveal a critical insight: **zero-shot confidence varies dramatically by input difficulty**.

- **Easy cases** (strongly positive/negative text): The model concentrates probability mass on the correct label. The logit gap is large, meaning the model has a strong internal signal.
- **Ambiguous cases** (neutral or mixed-sentiment text): Probability is spread more evenly across labels. The logit gap shrinks, and the model's "decision boundary" becomes noisy.

When you see **low confidence** (flat distribution, small logit gaps), it means the model's pretraining signal for this input is weak. This is exactly when you need to provide more guidance — either through:
- **Richer instructions** (detailed criteria for classification)
- **Few-shot examples** (showing the model what you expect)
- **Chain-of-thought reasoning** (letting the model reason step-by-step before classifying)

This natural transition from uncertain zero-shot to guided few-shot is what we'll explore after building more zero-shot intuition.

## 1. Direct Task Specification

In this section, we'll explore how to craft prompts that clearly define the task without providing examples. This is the essence of zero-shot prompting.

In [ ]:
def run_prompt(template, **kwargs):
    """Format a prompt template and generate a response."""
    prompt = template.format(**kwargs)
    messages = [{"role": "user", "content": prompt}]
    return generate(messages)

direct_task_prompt = """Classify the sentiment of the following text as positive, negative, or neutral.
Do not explain your reasoning, just provide the classification.

Text: {text}

Sentiment:"""

# Test the direct task specification
texts = [
    "I absolutely loved the movie! The acting was superb.",
    "The weather today is quite typical for this time of year.",
    "I'm disappointed with the service I received at the restaurant."
]

for text in texts:
    result = run_prompt(direct_task_prompt, text=text)
    print(f"Text: {text}")
    print(f"Sentiment: {result}")

## 2. Format Specification

Providing output format guidelines in the prompt can help structure the AI's response in a zero-shot scenario.

In [ ]:
format_spec_prompt = """Generate a short news article about {topic}.
Structure your response in the following format:

Headline: [A catchy headline for the article]

Lead: [A brief introductory paragraph summarizing the key points]

Body: [2-3 short paragraphs providing more details]

Conclusion: [A concluding sentence or call to action]"""

# Test the format specification prompting
topic = "The discovery of a new earth-like exoplanet"
result = run_prompt(format_spec_prompt, topic=topic)
print(result)

## 3. Multi-step Reasoning

For complex tasks, we can break them down into simpler zero-shot steps. This approach can improve the overall performance of the model.

In [ ]:
multi_step_prompt = """Analyze the following text for its main argument, supporting evidence, and potential counterarguments.
Provide your analysis in the following steps:

1. Main Argument: Identify and state the primary claim or thesis.
2. Supporting Evidence: List the key points or evidence used to support the main argument.
3. Potential Counterarguments: Suggest possible objections or alternative viewpoints to the main argument.

Text: {text}

Analysis:"""

# Test the multi-step reasoning approach
text = """While electric vehicles are often touted as a solution to climate change, their environmental impact is not as straightforward as it seems.
The production of batteries for electric cars requires significant mining operations, which can lead to habitat destruction and water pollution.
Moreover, if the electricity used to charge these vehicles comes from fossil fuel sources, the overall carbon footprint may not be significantly reduced.
However, as renewable energy sources become more prevalent and battery technology improves, electric vehicles could indeed play a crucial role in combating climate change."""

result = run_prompt(multi_step_prompt, text=text)
print(result)

## 4. Comparative Analysis

Let's compare different zero-shot prompt structures for the same task to evaluate their effectiveness.

In [ ]:
def compare_prompts(task, prompt_templates):
    """
    Compare different prompt templates for the same task.

    Args:
        task (str): The task description or input.
        prompt_templates (dict): A dictionary of prompt templates with their names as keys.
    """
    print(f"Task: {task}\n")
    for name, template in prompt_templates.items():
        result = run_prompt(template, task=task)
        print(f"{name} Prompt Result:")
        print(result)
        print("\n" + "-"*50 + "\n")

task = "Explain concisely the concept of blockchain technology"

prompt_templates = {
    "Basic": "Explain {task}.",
    "Structured": """Explain {task} by addressing the following points:
1. Definition
2. Key features
3. Real-world applications
4. Potential impact on industries"""
}

compare_prompts(task, prompt_templates)

## When Zero-Shot Fails

Zero-shot prompting has clear limits. When a task requires **specialized domain knowledge** — precise formats, codes, or conventions that appear rarely in pretraining data — the model's compressed representations simply don't contain enough signal to produce correct outputs. Let's see this in action.

In [ ]:
# Specialized domain tasks where zero-shot typically struggles
specialized_tasks = [
    {
        "name": "Medical Coding (ICD-10)",
        "expected": "J44.1 (COPD with acute exacerbation)",
        "prompt": (
            "Assign the correct ICD-10 diagnosis code for the following "
            "clinical note.\n\n"
            "Clinical Note: Patient presents with acute exacerbation of "
            "chronic obstructive pulmonary disease with lower respiratory "
            "infection.\n\nICD-10 Code:"
        ),
    },
    {
        "name": "Legal Citation (Bluebook Format)",
        "expected": "Brown v. Bd. of Educ., 347 U.S. 483 (1954)",
        "prompt": (
            "Convert the following case reference to proper Bluebook "
            "citation format.\n\n"
            "Case: The Supreme Court case Brown versus Board of Education "
            "decided in 1954, reported in volume 347 of the United States "
            "Reports starting at page 483.\n\nBluebook Citation:"
        ),
    },
    {
        "name": "Chemical IUPAC Naming",
        "expected": "2-acetoxybenzoic acid (aspirin)",
        "prompt": (
            "Provide the IUPAC systematic name for the molecule with this "
            "SMILES notation: CC(=O)Oc1ccccc1C(=O)O\n\nIUPAC Name:"
        ),
    },
    {
        "name": "Dewey Decimal Classification",
        "expected": "539.7548 (Quantum chromodynamics)",
        "prompt": (
            "Assign the correct Dewey Decimal Classification number for "
            "a book about quantum chromodynamics and quark confinement."
            "\n\nDewey Decimal Number:"
        ),
    },
]

for task in specialized_tasks:
    messages = [{"role": "user", "content": task["prompt"]}]
    result = generate(messages, max_new_tokens=100, do_sample=False)
    print(f"Task: {task['name']}")
    print(f"  Expected: {task['expected']}")
    print(f"  Model:    {result.strip()}")
    print("-" * 60)

In [ ]:
def show_entropy_and_top_tokens(prompt_text, top_k=10):
    """Measure prediction entropy and show top tokens — high entropy = confusion."""
    messages = [{"role": "user", "content": prompt_text}]
    full = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(full, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)

    # Shannon entropy in bits: higher = more uncertain
    log_probs = torch.log2(probs + 1e-10)
    entropy = -(probs * log_probs).sum().item()

    top_probs, top_indices = torch.topk(probs, top_k)
    top1 = top_probs[0].item()

    print(f"  Entropy: {entropy:.2f} bits  |  Top-1 prob: {top1:.4f}")
    for i, (p, idx) in enumerate(zip(top_probs, top_indices)):
        token = tokenizer.decode([idx.item()])
        bar = "\u2588" * int(p.item() * 40)
        print(f"    {i+1}. {token!r:<20} {p.item():.4f}  {bar}")
    print()

# --- Familiar task: low entropy, confident ---
print("=" * 60)
print("FAMILIAR TASK: Sentiment classification")
print("=" * 60)
show_entropy_and_top_tokens(
    "Classify as positive or negative: 'I love this!'\nAnswer:"
)

# --- Specialized task: high entropy, confused ---
print("=" * 60)
print("SPECIALIZED TASK: ICD-10 medical coding")
print("=" * 60)
show_entropy_and_top_tokens(
    "Assign the ICD-10 code for: acute exacerbation of COPD "
    "with lower respiratory infection.\nICD-10 Code:"
)

# --- Ambiguous task: moderate entropy ---
print("=" * 60)
print("AMBIGUOUS TASK: Open-ended classification")
print("=" * 60)
show_entropy_and_top_tokens(
    "Classify the following: 'The quarterly results met expectations.'"
    "\nCategory:"
)

### Why Zero-Shot Fails on Specialized Tasks

The entropy analysis above reveals the mechanism:

1. **Familiar tasks** → The model has seen thousands of similar prompt–completion pairs during pretraining. Probability concentrates on the correct label. Entropy is low.
2. **Specialized tasks** → The model has seen few (or no) examples of ICD-10 codes, Bluebook citations, or Dewey Decimal numbers in context. The probability distribution is **diffuse** — spread across many tokens — because the model doesn't have a strong learned mapping from the prompt pattern to the correct output format.
3. **Ambiguous tasks** → The model has partial signal but not enough to commit to a single answer. Entropy sits in the middle.

The core principle: **zero-shot performance is bounded by pretraining data coverage**. If the task pattern (instruction + expected output format) wasn't sufficiently represented in training data, the model cannot activate the right representations.

This directly motivates **few-shot learning** (the next notebook): by providing examples in the prompt, you *teach* the model the input–output mapping at inference time, bypassing the limitations of pretraining coverage.

## Zero-Shot Best Practices

Effective zero-shot prompting isn't just about asking the model to do something — it's about crafting prompts that activate the strongest possible learned representations. Below we demonstrate best practices across diverse task domains, then A/B test different framings for the same task.

In [ ]:
# Diverse zero-shot tasks showing best practices across domains
diverse_tasks = [
    {
        "name": "1. Sentiment Analysis (explicit labels + format constraint)",
        "messages": [
            {"role": "system", "content": (
                "You are a sentiment classifier. Respond with exactly "
                "one word: Positive, Negative, or Neutral."
            )},
            {"role": "user", "content": (
                "The concert was absolutely breathtaking, I've never "
                "experienced anything like it."
            )},
        ],
    },
    {
        "name": "2. Topic Classification (closed label set)",
        "messages": [
            {"role": "system", "content": (
                "Classify the text into exactly one category: Technology, "
                "Sports, Politics, Science, Entertainment. Reply with the "
                "category name only."
            )},
            {"role": "user", "content": (
                "The new CRISPR-Cas9 variant shows improved precision in "
                "editing mitochondrial DNA, opening doors for treating "
                "hereditary diseases."
            )},
        ],
    },
    {
        "name": "3. Named Entity Extraction (structured output)",
        "messages": [
            {"role": "system", "content": (
                "Extract all named entities from the text. Return a JSON "
                "object with keys: people, organizations, locations. "
                "Each key maps to a list of strings."
            )},
            {"role": "user", "content": (
                "Sundar Pichai announced at Google I/O in Mountain View "
                "that Alphabet would invest $2B in a new data center in "
                "Berlin, Germany."
            )},
        ],
    },
    {
        "name": "4. Language Detection (minimal output)",
        "messages": [
            {"role": "system", "content": (
                "Identify the language of the following text. Reply with "
                "the language name only."
            )},
            {"role": "user", "content": (
                "La vie est belle quand on prend le temps de regarder "
                "autour de soi."
            )},
        ],
    },
    {
        "name": "5. Code Explanation (concise technical summary)",
        "messages": [
            {"role": "system", "content":
                "Explain what this code does in one clear sentence."},
            {"role": "user", "content":
                "sorted(set(x for x in arr if x % 2 == 0), reverse=True)[:5]"},
        ],
    },
    {
        "name": "6. Intent Detection (pragmatic classification)",
        "messages": [
            {"role": "system", "content": (
                "Classify the user intent as one of: question, request, "
                "complaint, compliment, information. Reply with the "
                "intent only."
            )},
            {"role": "user", "content": (
                "Could you please reset my password? I've been locked "
                "out since yesterday."
            )},
        ],
    },
]

for task in diverse_tasks:
    result = generate(task["messages"], max_new_tokens=200, do_sample=False)
    print(f"--- {task['name']} ---")
    print(f"Result: {result}\n")

In [ ]:
# A/B Test: same task, five different framings — from vague to precise
text_to_classify = (
    "The battery life is impressive but the screen quality "
    "could be better."
)

framings = {
    "A) Vague framing": [
        {"role": "user", "content":
            f"What do you think about this: {text_to_classify}"},
    ],
    "B) Basic instruction": [
        {"role": "user", "content":
            f"Classify the sentiment: {text_to_classify}"},
    ],
    "C) Explicit labels": [
        {"role": "user", "content": (
            "Classify the sentiment of this text as exactly one of: "
            "Positive, Negative, or Mixed.\n\n"
            f"Text: {text_to_classify}\n\nSentiment:"
        )},
    ],
    "D) System + User separation": [
        {"role": "system", "content": (
            "You are a precise sentiment classifier. Respond with "
            "exactly one word: Positive, Negative, or Mixed."
        )},
        {"role": "user", "content": f"Text: {text_to_classify}"},
    ],
    "E) Detailed with structured output": [
        {"role": "system", "content": (
            "You are a sentiment analysis expert. Analyze the given "
            "text and provide:\n"
            "1. Overall sentiment (Positive/Negative/Mixed)\n"
            "2. Confidence (High/Medium/Low)\n"
            "3. Key phrase that determined your classification\n"
            "Respond in exactly this format."
        )},
        {"role": "user", "content": f"Text: {text_to_classify}"},
    ],
}

for name, messages in framings.items():
    result = generate(messages, max_new_tokens=150, do_sample=False)
    print(f"=== {name} ===")
    print(f"Result: {result}\n")

print("-" * 60)
print("Notice: More explicit framing produces more focused, structured")
print("responses. Vague framing lets the model ramble or interpret the")
print("task differently — because the attention pattern is less directed.")

## The Zero-Shot Spectrum

Zero-shot prompting is not binary — it exists on a **spectrum of instruction richness**:

| Level | Description | Example |
|-------|-------------|---------|
| **Bare zero-shot** | Minimal or no instruction | *"Classify: [text]"* |
| **Instructed zero-shot** | Clear task description with labels | *"Classify as positive or negative: [text]"* |
| **Rich zero-shot** | Detailed criteria, output format, role | *System role + classification criteria + format spec* |
| **Few-shot** | Examples provided in the prompt | *(next notebook)* |

Each step along the spectrum adds more **context tokens** that the attention mechanism can use to constrain output. More instruction detail means:
- **Sharper attention** on task-relevant features
- **Narrower output distribution** (higher confidence on correct tokens)
- **Better format compliance** (the model knows *what shape* the answer should take)

The practical implication: before reaching for few-shot examples, first try enriching your zero-shot instructions. Often, a well-crafted zero-shot prompt with detailed instructions can match or exceed a hastily-constructed few-shot prompt.

In [ ]:
# Progressive enhancement: same task at four instruction levels
task_text = (
    "The new policy will increase taxes for high earners while "
    "providing relief for middle-class families."
)

levels = [
    {
        "name": "Level 1 \u2014 Minimal instruction",
        "messages": [
            {"role": "user", "content": f"Classify: {task_text}"},
        ],
    },
    {
        "name": "Level 2 \u2014 Clear task with labels",
        "messages": [
            {"role": "user", "content": (
                "Classify the political leaning of this statement as "
                "Left, Right, or Center.\n\n"
                f"Text: {task_text}\n\nClassification:"
            )},
        ],
    },
    {
        "name": "Level 3 \u2014 Detailed instruction with criteria",
        "messages": [
            {"role": "system", "content": (
                "You are a political analyst. Classify text by political "
                "leaning:\n"
                "- Left: favors progressive taxation, social programs, "
                "government intervention\n"
                "- Right: favors lower taxes, free market, limited "
                "government\n"
                "- Center: balanced, moderate, or neutral position\n"
                "Respond with the classification and a one-sentence "
                "explanation."
            )},
            {"role": "user", "content": f"Text: {task_text}"},
        ],
    },
    {
        "name": "Level 4 \u2014 Rich instruction with output format",
        "messages": [
            {"role": "system", "content": (
                "You are a political analyst specializing in policy "
                "analysis. Classify text by political leaning using "
                "these criteria:\n\n"
                "- Left/Progressive: Supports progressive taxation, "
                "expanded social safety nets, government regulation, "
                "wealth redistribution\n"
                "- Right/Conservative: Supports flat/reduced taxation, "
                "free market solutions, limited government, individual "
                "responsibility\n"
                "- Center/Moderate: Balanced approach, compromise "
                "positions, pragmatic policy\n\n"
                "Respond in this exact format:\n"
                "Leaning: [Left/Right/Center]\n"
                "Confidence: [High/Medium/Low]\n"
                "Key signals: [phrases from the text]\n"
                "Reasoning: [one sentence]"
            )},
            {"role": "user", "content": (
                f"Analyze this policy statement:\n\n\"{task_text}\""
            )},
        ],
    },
]

for level in levels:
    result = generate(level["messages"], max_new_tokens=200, do_sample=False)
    print("=" * 60)
    print(level["name"])
    print("=" * 60)
    print(result)
    print()

print("Each level adds more context tokens for the attention mechanism,")
print("producing increasingly precise and well-formatted responses.")

## 📝 Key Takeaways

- **Overview** — revisit this section for reference
- **Motivation** — revisit this section for reference
- **Key Components** — revisit this section for reference
- **Method Details** — revisit this section for reference
- **Conclusion** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory